# 변수 설정

In [4]:
import pandas as pd

# 1. 원본 파일 불러오기
df = pd.read_excel("merged_reviews_updated_v2.xlsx")

# 2. 작성일과 넷플릭스 공개일을 숫자에서 날짜로 변환 (엑셀 기준 숫자면 아래처럼 처리)
df['작성일'] = pd.to_datetime(df['작성일'], errors='coerce', unit='D', origin='1899-12-30')
df['넷플릭스 공개일'] = pd.to_datetime(df['넷플릭스 공개일'], errors='coerce', unit='D', origin='1899-12-30')

# 3. 유효한 날짜만 필터링
df = df[df['작성일'].notna() & df['넷플릭스 공개일'].notna()]

# 4. 작성_월
df['작성_월'] = df['작성일'].dt.to_period('M').astype(str)

# 5. event_time (넷플릭스 공개일 기준 ±개월)
df['event_time'] = ((df['작성일'] - df['넷플릭스 공개일']) / pd.Timedelta(days=30)).round().astype("Int64")

# 6. 이벤트 더미 생성
for t in range(-3, 4):
    col = f"ev_{'m' if t < 0 else 'p'}{abs(t)}"
    df[col] = (df['event_time'] == t).fillna(False).astype(int)

# 7. treatment
df['treatment'] = (df['작성일'] >= df['넷플릭스 공개일']).astype(int)

# 8. 리뷰 수 집계
review_counts = df.groupby(['영화명', '작성_월']).size().reset_index(name='review_count')
df = df.merge(review_counts, on=['영화명', '작성_월'], how='left')

# 9. 엑셀 저장
df.to_excel("merged_reviews_updated_v3(fixed_dates).xlsx", index=False)


In [19]:
pd.to_numeric(df['별점'], errors='coerce')

0        10
1        10
2        10
3        10
4        10
         ..
76378     2
76379    10
76380    10
76381    10
76382     4
Name: 별점, Length: 76383, dtype: int64

# staggered did 회귀

In [9]:
import pandas as pd
import statsmodels.formula.api as smf

# 1. 데이터 불러오기
df = pd.read_excel("merged_reviews_updated_v3(fixed_dates).xlsx")

# 2. 조건에 맞게 필터링
# - OTT 출시 전(-3, -2, -1): 실관람객 == 1
# - OTT 출시 후(0, 1, 2, 3): OTT관람객 == 1
pre = df[(df['event_time'].isin([-3, -2, -1])) & (df['실관람객'] == 1)]
post = df[(df['event_time'].isin([0, 1, 2, 3])) & (df['OTT관람객'] == 1)]
df_reg = pd.concat([pre, post])
df_reg = df_reg[df_reg['별점'].notna()].copy()

# 3. 고정효과 변수 범주형 처리
df_reg['영화명'] = df_reg['영화명'].astype('category')
df_reg['작성_월'] = df_reg['작성_월'].astype('category')

# 4. 회귀식 (ev_m1 제외 → 기준 시점)
formula = '별점 ~ ev_m3 + ev_m2 + ev_p0 + ev_p1 + ev_p2 + ev_p3 + C(영화명) + C(작성_월)'

# 5. 회귀 실행 (영화명 기준 cluster robust SE)
model = smf.ols(formula=formula, data=df_reg).fit(cov_type='cluster', cov_kwds={'groups': df_reg['영화명']})

# 6. 결과 출력
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                     별점   R-squared:                       0.466
Model:                            OLS   Adj. R-squared:                  0.458
Method:                 Least Squares   F-statistic:                 1.615e-10
Date:                Thu, 31 Jul 2025   Prob (F-statistic):               1.00
Time:                        15:46:35   Log-Likelihood:                -36416.
No. Observations:               16187   AIC:                         7.333e+04
Df Residuals:                   15939   BIC:                         7.524e+04
Df Model:                         247                                         
Covariance Type:              cluster                                         
                                           coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------------


C:\Users\82104\anaconda3\envs\hanyang_paper\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 249, but rank is 2
  warnings.warn('covariance of constraints does not have full '
C:\Users\82104\anaconda3\envs\hanyang_paper\Lib\site-packages\statsmodels\regression\linear_model.py:1884: RuntimeWarning: invalid value encountered in sqrt
  return np.sqrt(np.diag(self.cov_params()))


# 장르 고정효과 추가

In [12]:
import pandas as pd
import statsmodels.formula.api as smf

# 1. 데이터 불러오기
df = pd.read_excel("merged_reviews_updated_v3(fixed_dates).xlsx")

# 2. 조건에 맞게 필터링
pre = df[(df['event_time'].isin([-3, -2, -1])) & (df['실관람객'] == 1)]
post = df[(df['event_time'].isin([0, 1, 2, 3])) & (df['OTT관람객'] == 1)]
df_reg = pd.concat([pre, post])
df_reg = df_reg[df_reg['별점'].notna()].copy()

# 3. 고정효과용 변수들을 범주형으로 변환
df_reg['영화명'] = df_reg['영화명'].astype('category')
df_reg['작성_월'] = df_reg['작성_월'].astype('category')
df_reg['장르'] = df_reg['장르'].astype('category')

# 4. 회귀식 (기준 시점 ev_m1 제외 + 장르 고정효과 추가)
formula = (
    '별점 ~ ev_m3 + ev_m2 + ev_p0 + ev_p1 + ev_p2 + ev_p3 '
    '+ C(영화명) + C(작성_월) + C(장르)'
)

# 5. 회귀 실행 (영화명 기준 cluster robust SE)
model = smf.ols(formula=formula, data=df_reg).fit(
    cov_type='cluster', cov_kwds={'groups': df_reg['영화명']}
)

# 6. 결과 출력
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                     별점   R-squared:                       0.465
Model:                            OLS   Adj. R-squared:                  0.457
Method:                 Least Squares   F-statistic:                  0.009700
Date:                Thu, 31 Jul 2025   Prob (F-statistic):               1.00
Time:                        15:51:40   Log-Likelihood:                -36435.
No. Observations:               16187   AIC:                         7.337e+04
Df Residuals:                   15939   BIC:                         7.527e+04
Df Model:                         247                                         
Covariance Type:              cluster                                         
                                           coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------------


C:\Users\82104\anaconda3\envs\hanyang_paper\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 315, but rank is 27
  warnings.warn('covariance of constraints does not have full '
C:\Users\82104\anaconda3\envs\hanyang_paper\Lib\site-packages\statsmodels\regression\linear_model.py:1884: RuntimeWarning: invalid value encountered in sqrt
  return np.sqrt(np.diag(self.cov_params()))


# CALLAWAY SANT;A

In [27]:
from csdid.att_gt import ATTgt

In [33]:
import pandas as pd
from csdid.att_gt import ATTgt

# 1) 데이터 불러오기 --------------------------------------------------------------
df = pd.read_excel("merged_reviews_updated_v3(fixed_dates).xlsx")

# 2) 기본 변수 -------------------------------------------------------------------
df["id"]     = df["영화명"]                 # (unit) 영화 단위 식별자
df["rating"] = df["별점"].astype(float)      # (y)   별점

# 3) period  : 작성일(연-월) → 순차적 번호 ---------------------------------------
df["period"] = (
    pd.to_datetime(df["작성일"], errors="coerce")
      .dt.to_period("M")               # → Period[M] (예: 2020-01)
      .astype("category")              # 카테고리로 변환해야 .cat.codes 사용 가능
      .cat.codes + 1                   # 1부터 시작하는 정수 (time index)
)

# 4) G (처치시기) : 영화별 넷플릭스 최초공개 연-월 → 번호 --------------------------
df["G"] = (
    pd.to_datetime(df["넷플릭스 공개일"], errors="coerce")
      .dt.to_period("M")
      .astype("category")
      .cat.codes + 1
)

# 결측(날짜 파싱 실패 등) 행 제거
df = df.dropna(subset=["period", "G", "rating"])

# 5) 패널 정렬(선택) --------------------------------------------------------------
df = df.sort_values(["id", "period"])

# 6) Callaway-Sant’Anna ATT(g,t) 추정 -------------------------------------------
att = (
    ATTgt(
        yname     = "rating",
        tname     = "period",
        idname    = "id",
        gname     = "G",
        data      = df,
        control_group = "notyettreated",   # never-treated 그룹이 없으므로
        panel     = True,
        allow_unbalanced_panel = True      # 리뷰 수 달라도 허용
    )
    .fit()
)

# 7) 그룹×시점 ATT 테이블 확인 ----------------------------------------------------
att_tbl = att.att_gt          # DataFrame (컬럼: group, time, att, se, ...)
print("\n— 일부 미리보기 —")
print(att_tbl.head())

# 필요하면 NaN 행 비율 확인
nan_cnt = att_tbl["att"].isna().sum()
print(f"\nATT 테이블 NaN 개수: {nan_cnt:,}")

# 8) 집계 ATT (simple 평균)  ------------------------------------------------------
agg = att.aggte(typec="simple", na_rm=True)   # ← NaN 제거 옵션
print("\n— 집계 ATT (simple) —")
print(agg.summary)

# 9) 선택: 이벤트 스터디 그림 ------------------------------------------------------
# from csdid.plots import event_study
# event_study(att)               # 주피터노트북이면 바로 그림이 뜸

Dropped 581 units that were already treated in the first period.
No units in group 3 in time period 2, e1
No units in group 3 in time period 1, e2
No units in group 3 in time period 3, e1
No units in group 3 in time period 2, e2
No units in group 3 in time period 4, e1
No units in group 3 in time period 3, e2
No units in group 3 in time period 5, e1
No units in group 3 in time period 4, e2
No units in group 3 in time period 6, e1
No units in group 3 in time period 5, e2
No units in group 3 in time period 7, e1
No units in group 3 in time period 6, e2
No units in group 3 in time period 8, e1
No units in group 3 in time period 7, e2
No units in group 3 in time period 9, e1
No units in group 3 in time period 8, e2
No units in group 3 in time period 10, e1
No units in group 3 in time period 9, e2
No units in group 3 in time period 11, e1
No units in group 3 in time period 10, e2
No units in group 3 in time period 12, e1
No units in group 3 in time period 11, e2
No units in group 3 in time 

C:\Users\82104\anaconda3\envs\hanyang_paper\Lib\site-packages\csdid\attgt_fnc\preprocess_did.py:412: UserWarning: Be aware that there are some small groups in your dataset.
  Check groups: 4,10,15,18,23,28,40.
  warnings.warn(f"Be aware that there are some small groups in your dataset.\n  Check groups: {gpaste}.")


No units in group 4 in time period 23, e1
No units in group 4 in time period 22, e2
No available control units for group 4 in time period 23, e3
No available control units for group 4 in time period 22, e4
No units in group 4 in time period 24, e1
No units in group 4 in time period 23, e2
No available control units for group 4 in time period 24, e3
No available control units for group 4 in time period 23, e4
No units in group 4 in time period 25, e1
No units in group 4 in time period 24, e2
No available control units for group 4 in time period 25, e3
No available control units for group 4 in time period 24, e4
No units in group 4 in time period 26, e1
No units in group 4 in time period 25, e2
No available control units for group 4 in time period 26, e3
No available control units for group 4 in time period 25, e4
No units in group 4 in time period 27, e1
No units in group 4 in time period 26, e2
No available control units for group 4 in time period 27, e3
No available control units for 

AttributeError: 'ATTgt' object has no attribute 'att_gt'

In [35]:
# (1) 그룹-시점별 관측치 수 집계
cnt_tbl = (
    df.groupby(['G', 'period'])      # G = 최초 OTT 공개 ‘그룹’, period = 연-월 번호
      .size()
      .rename('n')
      .reset_index()
)

print(cnt_tbl.head())               # 구조 확인

# (2) 피벗 → 한눈에 빈 셀 찾기
pivot = cnt_tbl.pivot(index='G', columns='period', values='n')\
               .fillna(0).astype(int)

# 예시: 그룹 47의 시점별 관측치
print(pivot.loc[47])

# (3) 어디가 비었는지 요약
missing_mat   = pivot == 0
empty_per_grp = missing_mat.sum(axis=1).sort_values(ascending=False)   # 그룹별 빈 셀 개수
empty_per_t   = missing_mat.sum(axis=0).sort_values(ascending=False)   # 시점별 빈 셀 개수


   G  period    n
0  1      34  420
1  1      35  115
2  1      36   15
3  1      37    8
4  1      38    1
period
1      0
2      0
3      0
4      0
5      0
      ..
95    20
96     7
97    10
98     4
99     2
Name: 47, Length: 99, dtype: int64


# 다시 이게 찐

# 1.파일 불러오기 & 날짜 컬럼 변환

In [30]:
from csdid.att_gt import ATTgt

In [3]:
import pandas as pd

FILE = "merged_reviews_updated_v2.xlsx"
df = pd.read_excel(FILE)

# ── Excel serial → datetime ───────────────────────────
for col in ["개봉일", "작성일", "넷플릭스 공개일"]:
    df[col] = pd.to_datetime(df[col], unit="D", origin="1899-12-30")

# 2.처치(TREATED)윈도우 정의

In [13]:
df["treat"] = (
    (df["OTT관람객"] == 1) &
    (df["작성일"] >= df["넷플릭스 공개일"]) &
    (df["작성일"] <= df["넷플릭스 공개일"] + pd.Timedelta(days=13))
).astype(int)

# 3. not-yet-treated(처치 전 대조) 정의

In [16]:
df["notyet"] = (
    (df["실관람객"] == 1) &
    (df["작성일"] >= df["개봉일"]) &
    (df["작성일"] <  df["넷플릭스 공개일"])
).astype(int)

# 4. 주(week) 단위 패널 재구성

In [19]:
# ── id : 영화 고유번호 ─────────────────────
df["id"] = df["영화명"].astype("category").cat.codes + 1   # 1,2,...

# ── g : 최초 처치시점(넷플릭스 공개 주) ───
df["G"] = df.groupby("id")["넷플릭스 공개일"].transform("first").dt.isocalendar().week

# ── t : 관측시점(작성 주) ───────────────────
df["period"] = df["작성일"].dt.isocalendar().week

# ── agg : 영화-주 평균 별점(가중평균 가능) ──
panel = (
    df.groupby(["id","G","period"], as_index=False)
      .agg(rating=("별점", "mean"),
           n_reviews=("별점", "size"),
           treat=("treat", "max"),    # 주 안에 한 건이라도 treat면 1
           notyet=("notyet", "max"))
)

# 5. csdid.ATTgt 예시

In [34]:
df['rating'] = df['별점'].astype(float)

In [48]:
print(raw.keys())

dict_keys(['group', 't', 'att', 'se', 'c'])


In [52]:
att = (
    ATTgt(
        yname   = "rating",
        tname   = "period",
        idname  = "id",
        gname   = "G",
        data    = df,
        panel   = True,
        allow_unbalanced_panel = True,
        control_group          = "notyettreated"   # ← 핵심
    )
    .fit()
)

# ───────────────────────────────────────────────────────────────
# 5) 결과 정리  (did_object → DataFrame)
# ───────────────────────────────────────────────────────────────
raw = att.did_object                # dict with numpy arrays
# key 이름은 버전에 따라 group / t / att / se / c 로 나옴
att_tbl = pd.DataFrame({
    "group" : raw["group"],
    "time"  : raw["t"],             # ← 't'가 시점
    "att"   : raw["att"],
    "se"    : raw["se"]
})

# ───────────────────────────────────────────────────────────────
# 6) NaN 셀 제거 후 통계 구하기
# ───────────────────────────────────────────────────────────────
att_tbl = att_tbl.dropna(subset=["att"])
mean_att      = att_tbl["att"].mean()
median_att    = att_tbl["att"].median()
weighted_att  = np.average(att_tbl["att"], weights=1/att_tbl["se"]**2)

print(f"▶ 평균 ATT (NaN 제외):        {mean_att:.4f}")
print(f"▶ 중앙값 ATT (NaN 제외):      {median_att:.4f}")
print(f"▶ 가중평균 ATT (1/SE² 가중):  {weighted_att:.4f}")

Dropped 966 units that were already treated in the first period.


C:\Users\82104\anaconda3\envs\hanyang_paper\Lib\site-packages\csdid\attgt_fnc\preprocess_did.py:412: UserWarning: Be aware that there are some small groups in your dataset.
  Check groups: 11,12,35,44.
  warnings.warn(f"Be aware that there are some small groups in your dataset.\n  Check groups: {gpaste}.")


No available control units for group 2 in time period 51, e3
No available control units for group 2 in time period 50, e4
No units in group 3 in time period 31, e1
No available control units for group 3 in time period 51, e3
No available control units for group 3 in time period 50, e4
No units in group 4 in time period 10, e1
No units in group 4 in time period 14, e1
No units in group 4 in time period 16, e1
No units in group 4 in time period 17, e1
No units in group 4 in time period 18, e1
No units in group 4 in time period 19, e1
No units in group 4 in time period 20, e1
No units in group 4 in time period 21, e1
No units in group 4 in time period 22, e1
No units in group 4 in time period 23, e1
No units in group 4 in time period 24, e1
No units in group 4 in time period 25, e1
No units in group 4 in time period 26, e1
No units in group 4 in time period 27, e1
No units in group 4 in time period 28, e1
No units in group 4 in time period 29, e1
No units in group 4 in time period 30, e1


NameError: name 'np' is not defined

# 한번 마지막으로 8월1일 세미나 전 마지막

In [55]:
# 1) 데이터 불러오기  ─ 파일명만 맞춰 주세요
# ───────────────────────────────────────────────────────────────
PATH = "merged_reviews_updated_v2.xlsx"  # ⬅︎ 업로드한 최신 파일
df = pd.read_excel(PATH)

# ───────────────────────────────────────────────────────────────
# 2) 기본 정리
# ───────────────────────────────────────────────────────────────
#   2-1. 평점 → 숫자
df["rating"] = pd.to_numeric(df["별점"], errors="coerce")

#   2-2. id  : 영화 단위 고유 ID (없으면 enumerate)
if "영화id" in df.columns:
    df["id"] = df["영화id"]
else:
    df["id"] = df["영화명"].factorize()[0] + 1   # 1,2,3,…

#   2-3. period : “작성일”을 월 단위(정수)로
period_idx = (
    pd.to_datetime(df["작성일"])
      .dt.to_period("M")
      .astype(str)
)
# 맨 처음 월이 1이 되도록 정규화
period_map = {m: i+1 for i, m in enumerate(sorted(period_idx.unique()))}
df["period"] = period_idx.map(period_map)

#   2-4. G : 영화별 ‘첫 넷플릭스 공개 월’
g_period = (
    pd.to_datetime(df["넷플릭스 공개일"])
      .dt.to_period("M")
      .astype(str)
      .map(period_map)
)
df["G"] = g_period

# ───────────────────────────────────────────────────────────────
# 3) (선택) 아주 관측치가 적은 영화 그룹은 삭제
# ───────────────────────────────────────────────────────────────
min_obs_per_group = 5
cnt_by_G = df.groupby("G")["id"].count()
small_G = cnt_by_G[cnt_by_G < min_obs_per_group].index
df = df.loc[~df["G"].isin(small_G)]

# ───────────────────────────────────────────────────────────────
# 4) Callaway-Sant’Anna DID (ATTgt)
# ───────────────────────────────────────────────────────────────
att = (
    ATTgt(
        yname   = "rating",
        tname   = "period",
        idname  = "id",
        gname   = "G",
        data    = df,
        panel   = True,
        allow_unbalanced_panel = True,
        control_group          = "notyettreated"   # ← 핵심
    )
    .fit()
)

# ───────────────────────────────────────────────────────────────
# 5) 결과 정리  (did_object → DataFrame)
# ───────────────────────────────────────────────────────────────
raw = att.did_object                # dict with numpy arrays
# key 이름은 버전에 따라 group / t / att / se / c 로 나옴
att_tbl = pd.DataFrame({
    "group" : raw["group"],
    "time"  : raw["t"],             # ← 't'가 시점
    "att"   : raw["att"],
    "se"    : raw["se"]
})

# ───────────────────────────────────────────────────────────────
# 6) NaN 셀 제거 후 통계 구하기
# ───────────────────────────────────────────────────────────────
att_tbl = att_tbl.dropna(subset=["att"])
mean_att      = att_tbl["att"].mean()
median_att    = att_tbl["att"].median()
weighted_att  = np.average(att_tbl["att"], weights=1/att_tbl["se"]**2)

print(f"▶ 평균 ATT (NaN 제외):        {mean_att:.4f}")
print(f"▶ 중앙값 ATT (NaN 제외):      {median_att:.4f}")
print(f"▶ 가중평균 ATT (1/SE² 가중):  {weighted_att:.4f}")

# ───────────────────────────────────────────────────────────────
# 7) 필요하면 CSV 저장
# ───────────────────────────────────────────────────────────────
# att_tbl.to_csv("attgt_by_group_time.csv", index=False)

dropped, 229, rows from original data due to missing data


ValueError: zero-size array to reduction operation maximum which has no identity

In [57]:
# (A) 필터 조건 완화 또는 제거
# min_obs_per_group = 1   # 그냥 1로 두면 사실상 필터를 안 쓰는 것과 같음
# 또는 아래 3줄을 통째로 주석 처리
# cnt_by_G = df.groupby("G")["id"].count()
# small_G = cnt_by_G[cnt_by_G < min_obs_per_group].index
# df = df.loc[~df["G"].isin(small_G)]

# (B) 필터가 꼭 필요하다면, 삭제 전에 몇 개 남는지 체크
cnt_by_G = df.groupby("G")["id"].count()
print("필터 전 그룹 수:", cnt_by_G.shape[0])
small_G = cnt_by_G[cnt_by_G < 5].index
df = df.loc[~df["G"].isin(small_G)]
print("필터 후 그룹 수:", df["G"].nunique())


필터 전 그룹 수: 1
필터 후 그룹 수: 1
